In [ ]:
import pandas as pd

In [ ]:
# Path to the template TSV file containing the headers
template_file_path = "../node_template/submission_case_template.tsv"

# Read the template TSV file to extract the headers
df_template = pd.read_csv(template_file_path, sep="\t", nrows=0)  # Read only the header
headers = df_template.columns.tolist()  # Extract the headers as a list

In [ ]:
# Observational Patients

# File path to the input CSV file
file_path_obs = "/Users/jinn/Documents/IU/ARDaC/DCC_data_release_v2.0.0/raw_data/Data for Nanxin/OBS Final Datasets/OBS_SUBJECTS.csv"

# Read the file using pandas
df_obs_input = pd.read_csv(file_path_obs, sep=",", dtype=str)
# 2025.5.23 Exclude the invalid patient with usubjid == "31014"
df_obs_input = df_obs_input[df_obs_input["usubjid"] != "31014"]

# Initialize a DataFrame with the defined headers and same index as df_obs_input
df_obs_output = pd.DataFrame(index=df_obs_input.index, columns=headers)

# Assign fixed values using .loc to align with the index
df_obs_output.loc[:, "*type"] = "case"
df_obs_output.loc[:, "project_id"] = "ARDaC-AlcHepNet"
df_obs_output.loc[:, "*studies.submitter_id"] = "obs"
df_obs_output.loc[:, "index_date"] = "Study Enrollment"

# Map dynamic values using .apply
df_obs_output["*submitter_id"] = df_obs_input["usubjid"].apply(lambda x: f"{x}_obs" if pd.notna(x) else None)
df_obs_output["cohort"] = df_obs_input["obs_arm"].apply(lambda x: x.split(":")[-1].strip() if pd.notna(x) else None)
df_obs_output["study_site"] = df_obs_input["site"].apply(lambda x: x.strip() if pd.notna(x) else None)
df_obs_output["vital_status"] = df_obs_input["ALIVE"].apply(lambda x: "alive" if x == "Y" else "dead" if x == "N" else None)

# Display the resulting DataFrame
print("Observational Patients Output DataFrame preview:")
print(df_obs_output.head())

In [ ]:
# 2025.05.22 V2.0.2 Add AKI to OBS patient
file_path_aki_obs = "/Users/jinn/Documents/IU/ARDaC/DCC_data_release_v2.0.0/raw_data/Data for Nanxin/OBS Final Datasets/OBS_ABX_AKI_CX.csv"
df_aki_obs = pd.read_csv(file_path_aki_obs, sep=",", dtype=str)

# Ensure 'akiaernyn' has a default of 'Unknown' for missing values
df_aki_obs["akiaernyn"] = df_aki_obs["akiaernyn"].fillna("Unknown")

# Build a mapping from usubjid to akiaernyn
aki_map_obs = df_aki_obs.set_index("usubjid")["akiaernyn"].to_dict()

# Assign aki_status: use map if present, else default to 'Unknown'
df_obs_output["aki_status"] = df_obs_input["usubjid"].apply(lambda x: aki_map_obs.get(x, "Unknown"))

In [ ]:
# 2025.05.28 V2.0.3 Add "days_to_aki", "days_to_death", 
#     "days_90_survival", "days_180_survival", "days_90_aki", "days_180_aki" to OBS patient

# 2025.06.18 V2.0.4 Update: Compare akions1dat, akions2dat, akions3dat and use the earliest as AKI onset

# Convert date columns
df_obs_input["obs_enrolldat"] = pd.to_datetime(df_obs_input["obs_enrolldat"], errors="coerce")
df_obs_input["dthdtc"] = pd.to_datetime(df_obs_input["dthdtc"], errors="coerce")
df_aki_obs["akions1dat"] = pd.to_datetime(df_aki_obs["akions1dat"], errors="coerce")
df_aki_obs["akions2dat"] = pd.to_datetime(df_aki_obs["akions2dat"], errors="coerce")
df_aki_obs["akions3dat"] = pd.to_datetime(df_aki_obs["akions3dat"], errors="coerce")

# Get earliest AKI date per subject across all three fields
aki_dates = df_aki_obs[['usubjid', 'akions1dat', 'akions2dat', 'akions3dat']].copy()
aki_dates["earliest_aki"] = aki_dates[["akions1dat", "akions2dat", "akions3dat"]].min(axis=1, skipna=True)
aki_dates = aki_dates.dropna(subset=["earliest_aki"])
aki_dates_map = aki_dates.set_index("usubjid")["earliest_aki"].to_dict()

# Calculate days_to_aki
df_obs_output["days_to_aki"] = df_obs_input["usubjid"].apply(
    lambda x: (aki_dates_map[x] - df_obs_input.loc[df_obs_input["usubjid"] == x, "obs_enrolldat"].values[0]).days
    if x in aki_dates_map and pd.notnull(df_obs_input.loc[df_obs_input["usubjid"] == x, "obs_enrolldat"].values[0]) else None
)

# Calculate days_to_death
df_obs_output["days_to_death"] = (
    df_obs_input["dthdtc"] - df_obs_input["obs_enrolldat"]
).dt.days

# Define classification functions
def classify_survival(days_to_death, cutoff):
    if pd.isna(days_to_death):
        return "Unknown"
    return "dead" if days_to_death <= cutoff else "alive"

def classify_aki(days_to_aki, lower, upper=None):
    if pd.isna(days_to_aki):
        return "Unknown"
    if upper is None:
        return "Yes" if days_to_aki <= lower else "No"
    if lower < days_to_aki <= upper:
        return "Yes"
    return "No" if days_to_aki > upper else "Unknown"

# Add survival status flags
df_obs_output["days_90_survival"] = df_obs_output["days_to_death"].apply(lambda x: classify_survival(x, 90))
df_obs_output["days_180_survival"] = df_obs_output["days_to_death"].apply(lambda x: classify_survival(x, 180))

# Add AKI occurrence flags
df_obs_output["days_90_aki"] = df_obs_output["days_to_aki"].apply(lambda x: classify_aki(x, 90))
df_obs_output["days_180_aki"] = df_obs_output["days_to_aki"].apply(lambda x: classify_aki(x, 90, 180))

# Final preview
print("Updated df_obs_output preview with days_to_aki and days_to_death:")
print(df_obs_output[["*submitter_id", "days_to_aki", "days_to_death"]].head())

In [ ]:
# Clinical Trial Patients
# File path to the RCT_SUBJECTS.csv file
file_path_rct = "/Users/jinn/Documents/IU/ARDaC/DCC_data_release_v2.0.0/raw_data/Data for Nanxin/RCT Final Datasets/RCT_SUBJECTS.csv"

# Read the RCT_SUBJECTS.csv file
df_input_rct = pd.read_csv(file_path_rct, sep=",", dtype=str)

# Initialize a new DataFrame with the defined headers and the same index as df_input_rct
df_rct_output = pd.DataFrame(index=df_input_rct.index, columns=headers)

# Assign fixed values using .loc for proper alignment
df_rct_output.loc[:, "*type"] = "case"
df_rct_output.loc[:, "project_id"] = "ARDaC-AlcHepNet"
df_rct_output.loc[:, "*studies.submitter_id"] = "clinical"
df_rct_output.loc[:, "index_date"] = "Study Enrollment"

# Map dynamic values based on the input file
df_rct_output["*submitter_id"] = df_input_rct["usubjid"].apply(lambda x: f"{x}_clinical" if pd.notna(x) else None)
df_rct_output["actarm"] = df_input_rct["rct_arm"].apply(lambda x: x.strip() if pd.notna(x) else None)
df_rct_output["rct_meld_strata"] = df_input_rct["rct_meld_strata"].apply(lambda x: x.strip() if pd.notna(x) else None)
df_rct_output["study_site"] = df_input_rct["site"].apply(lambda x: x.strip() if pd.notna(x) else None)
df_rct_output["vital_status"] = df_input_rct["ALIVE"].apply(lambda x: "alive" if x == "Y" else "dead" if x == "N" else None)

# Fill other unmapped columns with NaN for consistency
for col in df_rct_output.columns:
    if col not in ["*type", "project_id", "*submitter_id", "*studies.submitter_id", "actarm", 
                   "rct_meld_strata", "study_site", "vital_status", "index_date"]:
        df_rct_output[col] = None

# Display the first few rows of the output DataFrame for verification
print("Output RCT DataFrame preview:")
print(df_rct_output.head())

In [ ]:
# 2025.05.22 V2.0.2 Add AKI to RCT patient
file_path_ae = "/Users/jinn/Documents/IU/ARDaC/DCC_data_release_v2.0.0/raw_data/Data for Nanxin/RCT Final Datasets/RCT_ADVERSE_EVENTS.csv"
df_ae = pd.read_csv(file_path_ae, sep=",", dtype=str)

# Filter patients with ae_aki_indicator == "1"
aki_patients = df_ae[df_ae["ae_aki_indicator"] == "1"]["usubjid"].unique()

# Assign 'yes' to patients with AE=1, and 'no' to all others
df_rct_output["aki_status"] = df_input_rct["usubjid"].apply(lambda x: "Yes" if x in aki_patients else "No")

In [ ]:
# 2025.05.28 V2.0.3 Add "days_to_aki", "days_to_death", 
#     "days_90_survival", "days_180_survival", "days_90_aki", "days_180_aki" to RCT patient
# ----------------------------
# Add Days to AKI and Death for RCT
# ----------------------------

# Load AKI date from RCT_AKI_ADD.csv
file_path_aki_rct = "/Users/jinn/Documents/IU/ARDaC/DCC_data_release_v2.0.0/raw_data/Data for Nanxin/RCT Final Datasets/RCT_AKI_ADD.csv"
df_aki_rct = pd.read_csv(file_path_aki_rct, sep=",", dtype=str)

# Convert relevant dates
df_input_rct["rct_enrollrctdat"] = pd.to_datetime(df_input_rct["rct_enrollrctdat"], errors="coerce")
df_input_rct["dthdtc"] = pd.to_datetime(df_input_rct["dthdtc"], errors="coerce")
df_aki_rct["aki_date"] = pd.to_datetime(df_aki_rct["aki_date"], errors="coerce")

# Get earliest AKI date per subject
aki_dates_rct = df_aki_rct[['usubjid', 'aki_date']].dropna()
aki_dates_rct = aki_dates_rct.groupby('usubjid').agg({'aki_date': 'min'}).reset_index()
aki_dates_map_rct = aki_dates_rct.set_index("usubjid")["aki_date"].to_dict()

# Calculate days_to_aki
df_rct_output["days_to_aki"] = df_input_rct["usubjid"].apply(
    lambda x: (aki_dates_map_rct[x] - df_input_rct.loc[df_input_rct["usubjid"] == x, "rct_enrollrctdat"].values[0]).days
    if x in aki_dates_map_rct and pd.notnull(df_input_rct.loc[df_input_rct["usubjid"] == x, "rct_enrollrctdat"].values[0]) else None
)

# Calculate days_to_death
df_rct_output["days_to_death"] = (
    df_input_rct["dthdtc"] - df_input_rct["rct_enrollrctdat"]
).dt.days

# ----------------------------
# Add Flags for 90/180 Survival and AKI
# ----------------------------

def classify_survival(days_to_death, cutoff):
    if pd.isna(days_to_death):
        return "Unknown"
    return "dead" if days_to_death <= cutoff else "alive"

def classify_aki(days_to_aki, lower, upper=None):
    if pd.isna(days_to_aki):
        return "Unknown"
    if upper is None:
        return "Yes" if days_to_aki <= lower else "No"
    if lower < days_to_aki <= upper:
        return "Yes"
    return "No" if days_to_aki > upper else "Unknown"

df_rct_output["days_90_survival"] = df_rct_output["days_to_death"].apply(lambda x: classify_survival(x, 90))
df_rct_output["days_180_survival"] = df_rct_output["days_to_death"].apply(lambda x: classify_survival(x, 180))

df_rct_output["days_90_aki"] = df_rct_output["days_to_aki"].apply(lambda x: classify_aki(x, 90))
df_rct_output["days_180_aki"] = df_rct_output["days_to_aki"].apply(lambda x: classify_aki(x, 90, 180))

In [ ]:
# Export the "CASE" node.

# Save the Combined DataFrame to TSV
rct_output_path = "case_rct_DCC_data_release_v2-0-4.tsv"
obs_output_path = "case_obs_DCC_data_release_v2-0-4.tsv"

df_rct_output.to_csv(rct_output_path, sep="\t", index=False, header=True)
print(f"RCT patients file saved as: {rct_output_path}")
df_obs_output.to_csv(obs_output_path, sep="\t", index=False, header=True)
print(f"Observational patients file saved as: {obs_output_path}")